In [1]:
words = open("names.txt","r").read().splitlines()

In [2]:
stoi = {ch: i for i,ch in enumerate(sorted(set(".".join(words))))}
itos = {i:ch for ch,i in stoi.items()}

In [3]:
import torch

In [4]:
context = []
context_len = 3
iy = []
for w in words:
    w = w+'.'
    cntxt = [0]*context_len
    for ch in w:
        context.append(cntxt)
        iy.append(stoi[ch])
        cntxt = cntxt[1:] + [stoi[ch]]
context = torch.tensor(context)
iy = torch.tensor(iy)
ix = context
context.shape, iy.shape

(torch.Size([228146, 3]), torch.Size([228146]))

In [5]:
emb_len = 15
cntx_len = 3



In [6]:
import torch

C = torch.rand((27,emb_len)).float()
W1 = torch.randn((emb_len*cntx_len, 100)).float()*(5/3)/torch.sqrt(torch.tensor(emb_len*cntx_len)).item()
b1 = torch.randn((100,)).float() * 0.01

W2 = torch.randn((100, 27)).float()*0.01
b2 = torch.randn((27,)).float()*0.01

running_mean = torch.zeros((1,100),dtype=torch.float32);
running_std = torch.ones((1,100),dtype=torch.float32);

gamma = torch.ones((1,100),dtype=torch.float32)
beta = torch.zeros((1,100),dtype=torch.float32)

parameters= [C,W1,W2,b1,b2,gamma,beta]

for p in parameters:
    p.requires_grad = True

sum(p.nelement() for p in parameters)

7932

In [7]:
lli = torch.linspace(0,1,1000)
lossi = []

In [8]:
import torch.nn.functional as F

for i in range(20000):

    batch = torch.randint(0, iy.shape[0], (32,))
    emb = C[ix[batch]]
    h = emb.view(-1,emb_len*cntx_len) @ W1 + b1
    h = torch.tanh(h)

    mean_h = h.mean(dim = 0,keepdim=True)
    std_h = h.std(dim = 0, keepdim=True)

    h = gamma * (h - mean_h)/(std_h + 1e-5) + beta

    with torch.no_grad():
        running_mean = 0.99*running_mean + 0.01*mean_h
        running_std = 0.99*running_std + 0.01*std_h
    logits = h @ W2+ b2

    loss = F.cross_entropy(logits, iy[batch])

    for p in parameters:
        p.grad = None
        
    lr = 0.1 if i < 100000 else 0.01
    print(loss.item())
    loss.backward()
    for p in parameters:
        p.data -= lr*p.grad
loss

3.3139820098876953
3.276272773742676
3.205725908279419
3.2339792251586914
3.2458362579345703
3.2079813480377197
3.1451706886291504
3.2183637619018555
2.957170009613037
2.9538779258728027
3.0836899280548096
3.052966833114624
2.891641616821289
2.97568416595459
3.076140880584717
3.184898853302002
2.805873155593872
2.785395860671997
2.8923070430755615
3.0831310749053955
3.0784151554107666
3.2597663402557373
3.0370681285858154
2.774620532989502
2.937086582183838
2.952108383178711
2.886948823928833
2.8021697998046875
3.211366891860962
2.866018295288086
2.9399466514587402
2.9234256744384766
2.9056782722473145
2.858872652053833
2.7566256523132324
2.7319984436035156
3.057907819747925
2.828662395477295
2.725823163986206
2.9052321910858154
2.7060675621032715
2.7659482955932617
2.9594950675964355
2.9778473377227783
2.737534761428833
2.635805130004883
2.845513343811035
2.828385829925537
2.8192107677459717
3.2402303218841553
2.8483965396881104
2.8451130390167236
2.6656076908111572
2.759136438369751


tensor(1.9930, grad_fn=<NllLossBackward0>)

In [9]:
mean_h1 = torch.zeros((1,100)).float()
std_h2 = torch.zeros((1,100)).float()

with torch.no_grad():
    input = C[ix]

    h = input.view(-1,emb_len*cntx_len)@ W1 + b1
    h = torch.tanh(h)
    print(h)
    mean_h1 += h.mean(0,keepdim = True)
    std_h2 += h.std(0, keepdim = True)


tensor([[ 0.0775, -0.0697, -0.3416,  ...,  0.8103,  0.9921,  0.8902],
        [-0.8749, -0.9220, -0.6384,  ..., -0.6986,  0.9128, -0.1404],
        [-0.9313, -0.7193, -0.8852,  ...,  0.9930,  0.9996,  0.9923],
        ...,
        [ 0.6894, -0.9881,  0.7314,  ...,  0.5843, -0.7823,  0.3955],
        [-0.1159, -0.6778, -0.9657,  ...,  0.9999,  0.0889,  0.7202],
        [ 0.3818, -0.4632, -0.7010,  ...,  0.9761,  0.2817,  0.9449]])


In [10]:
running_std,std_h2

(tensor([[0.5426, 0.5517, 0.5479, 0.4356, 0.3546, 0.4616, 0.6956, 0.3155, 0.4819,
          0.6160, 0.5987, 0.5198, 0.6979, 0.6769, 0.6591, 0.4626, 0.4859, 0.5439,
          0.3215, 0.4443, 0.3362, 0.5870, 0.7091, 0.5846, 0.3127, 0.3807, 0.4064,
          0.2477, 0.5098, 0.4008, 0.6145, 0.4845, 0.5597, 0.4996, 0.1690, 0.2653,
          0.5568, 0.6312, 0.5135, 0.6202, 0.6915, 0.6126, 0.2702, 0.3545, 0.5213,
          0.6065, 0.3528, 0.2451, 0.2741, 0.6796, 0.6936, 0.4901, 0.6502, 0.3780,
          0.4669, 0.5914, 0.4848, 0.3195, 0.6156, 0.6333, 0.4471, 0.6798, 0.7013,
          0.5661, 0.2837, 0.4068, 0.6493, 0.4620, 0.5543, 0.6164, 0.5961, 0.5718,
          0.5896, 0.4310, 0.6691, 0.2978, 0.4668, 0.5069, 0.3265, 0.6027, 0.5172,
          0.5265, 0.5884, 0.5451, 0.3625, 0.7260, 0.5572, 0.3861, 0.3743, 0.6397,
          0.5389, 0.4009, 0.5454, 0.3901, 0.2874, 0.6893, 0.5499, 0.6221, 0.3896,
          0.3311]]),
 tensor([[0.5655, 0.5630, 0.5622, 0.4374, 0.3497, 0.4218, 0.6913, 0.3052, 0.4

In [11]:
for i in range(10):
    out  = ""
    cntxt = [0] * cntx_len
    with torch.no_grad():
        while True:
            input = C[cntxt]
            h = input.view(-1,emb_len*cntx_len)@ W1 + b1
            # print(h)
            logits = h@W2 + b2

            prob = torch.softmax(logits, dim = 1)

            ch_int = torch.multinomial(prob, num_samples = 1,).item()
            out += itos [ch_int]

            cntxt = cntxt[1:] + [ch_int]
            if ch_int == 0:
                break

        print(out)

zzhryzzhwh.
zhmyn.
jwwwww.
murryzz.
zzhwwwwwwwww.
zziry.
zzzahyn.
zwwan.
zzryj.
muggw.


In [233]:
input = torch.randn((500,10), dtype=torch.float32)
out = torch.tanh(input)

In [234]:
input.std(dim = 0,keepdim=True)

tensor([[1.0205, 0.9496, 1.0091, 0.9768, 0.9781, 1.0317, 0.9839, 1.0100, 0.9548,
         1.0128]])

In [235]:
out.std(dim = 0 ,keepdim=True)

tensor([[0.6385, 0.5998, 0.6139, 0.6190, 0.6293, 0.6272, 0.6145, 0.6351, 0.6179,
         0.6315]])

In [236]:
out

tensor([[ 0.6977, -0.5661, -0.8247,  ..., -0.4644,  0.7061, -0.4721],
        [-0.8357, -0.9172, -0.4342,  ..., -0.4698, -0.8648,  0.8850],
        [-0.9540,  0.9941, -0.4373,  ..., -0.5580, -0.8709, -0.5818],
        ...,
        [ 0.5995, -0.2308, -0.7539,  ...,  0.0013,  0.1129, -0.5778],
        [-0.8042, -0.4804, -0.3652,  ..., -0.3577,  0.3221,  0.5688],
        [ 0.7654, -0.1273, -0.4399,  ..., -0.1711, -0.0386,  0.5846]])

## Torchfiying the changese made in the MLP_practise_tochify.ipynb

In [154]:
class Linear:
    def __init__(self,input_dim, output_dim, bias = True):
        self.W = torch.randn((input_dim,output_dim),dtype = torch.float32)/ input_dim**0.5
        self.b = torch.randn((output_dim),dtype = torch.float32)*0.001 if bias else None

    def __call__(self,x):
        self.out = x @ self.W
        if self.b is not None:
            self.out += self.b
        return self.out
    
    def parameters(self):
        return [self.W] + [self.b] if self.b is not None else []

In [155]:
class Tanh:
    def __call__(self,x):
        self.out = torch.tanh(x)
        return self.out
    
    def parameters(self):
        return []


In [156]:
class Batch_Normalization:
    def __init__(self, dim, momentum = 0.1, eps = 1e-5):
        self.gamma = torch.ones((1,dim),dtype = torch.float32)
        self.beta = torch.zeros((1,dim),dtype = torch.float32)

        self.running_mean = torch.zeros((1,dim), dtype = torch.float32)
        self.running_var = torch.ones((1,dim), dtype = torch.float32)

        self.training = True
        self.momentum = momentum
        self.eps = eps
    
    def __call__(self, x):

        if self.training:
            mean = x.mean(dim = 1, keepdim = True)
            var = x.var(dim = 1, keepdim = True)
            with torch.no_grad():
                self.running_mean = (1 - self.momentum)*self.running_mean + self.momentum*mean 
                self.running_var = (1 - self.momentum)*self.running_var + self.momentum*var
        else:
            mean = self.running_mean
            var  = self.running_var

        self.out = self.gamma*(x - mean)/ torch.sqrt(var + self.eps) + self.beta

        return self.out

    def parameters(self):
        return [self.gamma, self.beta]



In [157]:
n_hidden = 100
input_dim = emb_len*context_len
vocab_size = 27
batch_size = 32

In [176]:
layers = [
    Linear(input_dim,n_hidden),Tanh(),
    Linear(n_hidden,n_hidden),Tanh(),
    Linear(n_hidden,n_hidden),Tanh(),
    Linear(n_hidden,n_hidden),Tanh(),
    Linear(n_hidden,vocab_size)
]

In [177]:
parameters = [C] + [p for layer in layers for p in layer.parameters()]


In [178]:
import torch.nn.functional as F
for layer in layers[:-1]:
    if isinstance(layer, Linear):
        layer.W *= 5/3
layers[-1].W *= 0.1

In [179]:
for p in parameters:
    p.requires_grad = True

In [180]:
for i in range(20000):
    batch = torch.randint(0,ix.shape[0],(batch_size,))
    x = C[ix[batch]]
    x = x.view(-1,input_dim)
    
    y = iy[batch]
    for layer in layers:
        x = layer(x)

    for p in parameters:
        p.grad = None
    

    loss = F.cross_entropy(x,y)

    for p in parameters:
        p.grad = None
    loss.backward()
    lr = 0.1
    for p in parameters:
        p.data += -lr*p.grad
    
    print(loss.item())
    

3.301642417907715
3.2602956295013428
3.164705753326416
3.24151611328125
3.137241840362549
3.0667641162872314
2.8969733715057373
2.9361329078674316
2.7302544116973877
2.9523158073425293
3.002925395965576
3.0544793605804443
2.774620771408081
2.938467502593994
2.742764949798584
2.76133131980896
2.719257354736328
2.6390016078948975
2.8983116149902344
2.8574376106262207
2.7021923065185547
2.8610427379608154
2.644850015640259
2.7885682582855225
2.9386112689971924
2.4813318252563477
2.7198355197906494
2.5874080657958984
2.9690520763397217
2.279340982437134
2.726972818374634
2.703057050704956
2.402872323989868
2.597712993621826
2.712902545928955
2.8495821952819824
2.6590168476104736
2.422889471054077
2.4034268856048584
2.444499969482422
2.7065703868865967
2.4585044384002686
2.327526807785034
2.744257688522339
2.601086378097534
2.4200291633605957
2.598308563232422
2.607086658477783
2.6013693809509277
2.781714677810669
2.471102476119995
2.540027379989624
2.715782403945923
2.446443557739258
2.709